# 🔥 Advanced · 3D Gaussian Splatting (official)

> 🔥 **Advanced / heavy lab.** Your photos → COLMAP poses → trained 3D Gaussians (real CUDA rasterizer).
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **25–45 min (CUDA build + COLMAP)** including downloads. Built on the official **[graphdeco-inria/gaussian-splatting](https://github.com/graphdeco-inria/gaussian-splatting)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Maps to lesson **B7 (3D Gaussian Splatting)** — the real 3D version of the 2D-splatting lab you trained from scratch.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — small scene, 7k iters | 1× RTX 3090/4090 or A100 24–40 GB — large scenes, 30k iters |
| **Storage** | photos + COLMAP + .ply ~ 1–3 GB | ~5–20 GB / scene; .ply 50–400 MB |
| **Time** | 7k iters ~ 7–15 min (+ COLMAP minutes) | 30k iters ~ 30–60 min / scene on a 3090/A100 |

**Full pipeline (scale-up):** `python train.py -s data/scene -m out --iterations 30000` (COLMAP cost grows with photo count).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi

## 1 · Clone (with submodules) and build the CUDA rasterizer

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd gaussian-splatting
!pip install -q plyfile tqdm
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

## 2 · Your photos → camera poses with COLMAP
Upload 20–200 photos of **one** object/scene (good overlap, static scene) into `data/scene/input/`, then:

In [ ]:
!apt-get -qq install -y colmap
!python convert.py -s data/scene

## 3 · Train the Gaussians

In [ ]:
!python train.py -s data/scene -m output/scene --iterations 7000 --test_iterations 7000 30000

## 4 · Render novel views
The interactive SIBR viewer needs a desktop GPU, so render images here instead (or download `output/scene/point_cloud/.../point_cloud.ply` and drop it into a web viewer such as antimatter15/splat).

In [ ]:
!python render.py -m output/scene
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
imgs = sorted(glob.glob("output/scene/train/ours_7000/renders/*.png"))[:3]
fig, ax = plt.subplots(1, max(1, len(imgs)), figsize=(4*max(1,len(imgs)), 4))
for a, p in zip(ax if len(imgs) > 1 else [ax], imgs):
    a.imshow(mpimg.imread(p)); a.axis("off")
plt.show()

## Evaluate — PSNR / SSIM / LPIPS on held-out views
3DGS holds out every 8th image as a test view; render and score them:

In [ ]:
!python render.py -m output/<your-scene>           # render train + held-out test views
!python metrics.py -m output/<your-scene>          # prints SSIM, PSNR, LPIPS over the test split

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
Gaussian scenes are the map substrate for **D · Scene / world** (SLAM, world models).

## Troubleshooting & next steps
- **CUDA build errors**: make sure the GPU runtime is on *before* the `pip install submodules/...` step; the build targets the runtime's CUDA.
- **Easier alternative**: `pip install nerfstudio && ns-train splatfacto` (the next lab) or the pip-installable **gsplat** rasterizer — no manual CUDA build.
- View/share the `.ply` in the browser; train longer (30k iters) for sharper results.